In [1]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def rekomendasi_produk(user_input, produk, daftar_produk):
    produk_input = daftar_produk[produk.index(user_input)] # Mengambil vektor produk yang dipilih
    similarity_scores = cosine_similarity([produk_input], daftar_produk) # Menghitung kesamaan kosinus
    
    rekomendasi_idx = np.argsort(similarity_scores[0])[:-1] # Hindari produk yang sama
    rekomendasi_idx = rekomendasi_idx[-2:][::-1]
    
    rekomendasi = [produk[i] for i in rekomendasi_idx]
    
    return rekomendasi

# Daftar produk dan fitur
produk = ["Laptop", "Smartphone", "Tablet", "Headphone"]
daftar_produk = np.array([
    [0.8, 0.2, 0.1], # Laptop
    [0.7, 0.1, 0.4], # Smartphone
    [0.6, 0.4, 0.3], # Tablet
    [0.1, 0.8, 0.5]  # Headphone
])

# Input pengguna
input_pengguna = "Smartphone"
hasil = rekomendasi_produk(input_pengguna, produk, daftar_produk)

print("Rekomendasi untuk", input_pengguna, "adalah:", hasil)

Rekomendasi untuk Smartphone adalah: ['Laptop', 'Tablet']


In [2]:
import pandas as pd

# Memuat dataset produk
data = pd.read_csv('produk_dataset_100.csv')
print(data.head())

   ID Produk   Nama Produk      Kategori  \
0          1        Topi C       Pakaian   
1          2        Oven X  Rumah Tangga   
2          3     Raket Pro      Olahraga   
3          4       Jaket X       Pakaian   
4          5  Keyboard Pro      Aksesori   

                                  Deskripsi  Harga Rating  
0         Topi terbaru untuk pengguna aktif   1688    3,9  
1       Oven handal untuk gaya hidup modern   1737    4,3  
2     Raket populer untuk gaya hidup modern   1005    3,4  
3         Jaket handal untuk pengguna aktif   1634      4  
4  Keyboard terbaik untuk gaya hidup modern   1640      5  


In [3]:
# Menghapus kolom yang tidak diperlukan
data = data[['ID Produk', 'Nama Produk', 'Deskripsi', 'Kategori', 'Rating']]

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Inisialisasi TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# Menghitung TF-IDF untuk deskripsi produk
tfidf_matrix = vectorizer.fit_transform(data['Deskripsi'])

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_product(user_query):
    # Menghitung TF-IDF untuk query pengguna
    query_tfidf = vectorizer.transform([user_query])
    
    # Menghitung cosine similarity
    similarity_scores = cosine_similarity(query_tfidf, tfidf_matrix)
    
    # Menemukan indeks produk dengan skor kemiripan tertinggi
    ranked_indices = similarity_scores.argsort()[0][::-1]
    
    # Menampilkan 5 rekomendasi teratas
    print("Rekomendasi Produk:")
    for i in range(5):
        product_index = ranked_indices[i]
        print(f"{data['Nama Produk'][product_index]} - Rating: {data['Rating'][product_index]}")

# Contoh input pengguna
recommend_product("smartphone untuk gaming")

Rekomendasi Produk:
Smartphone C - Rating: 3,9
Smartphone C - Rating: 3,8
Smartphone C - Rating: 4,8
Smartphone Pro - Rating: 4,8
Tablet Pro - Rating: 3,1


In [6]:
# Mengambil input dari pengguna
user_input = input("Cari produk berdasarkan deskripsi: ")
recommend_product(user_input)

Rekomendasi Produk:
Laptop A - Rating: 4,8
Laptop A - Rating: 3,9
Helm C - Rating: 3,8
Helm A - Rating: 5
Blender A - Rating: 4,2


In [ ]:
import numpy as np

# Fungsi untuk menghitung Precision@K
def precision_at_k(recommended_items, relevant_items, k):
    """
    Menghitung Precision@K.

    Parameters:
    recommended_items (list): Daftar item yang direkomendasikan (terurut berdasarkan skor).
    relevant_items (list): Daftar item yang relevan.
    k (int): Jumlah item yang dipertimbangkan untuk Precision@K.

    Returns:
    float: Nilai Precision@K.
    """
    recommended_items_at_k = recommended_items[:k] # Ambil hanya K item teratas
    relevant_and_recommended = set(recommended_items_at_k).intersection(relevant_items)
    return len(relevant_and_recommended) / k

# Contoh data
recommended_items = [10, 20, 30, 40, 50] # Item yang direkomendasikan (dalam urutan skor tertinggi)
relevant_items = [20, 50, 60]            # Item yang relevan (label ground truth)

# Menentukan nilai K
k = 3

# Hitung Precision@K
precision = precision_at_k(recommended_items, relevant_items, k)
print(f"Precision@{k}: {precision:.2f}")

Precision@3: 0.33


In [8]:
from sklearn.neighbors import NearestNeighbors

# Inisialisasi KNN untuk Collaborative Filtering
knn = NearestNeighbors(metric='cosine', algorithm='brute')
knn.fit(tfidf_matrix)

# Membangun fungsi rekomendasi dengan KNN
def recommend_with_knn(user_query, k=5):
    query_tfidf = vectorizer.transform([user_query])
    distances, indices = knn.kneighbors(query_tfidf, n_neighbors=k)
    
    print("Rekomendasi Produk berdasarkan KNN:")
    for i in range(k):
        product_index = indices[0][i]
        print(f"{data['Nama Produk'][product_index]} - Rating: {data['Rating'][product_index]}")

# Contoh penggunaan KNN
recommend_with_knn("laptop dengan baterai tahan lama", k=5)

Rekomendasi Produk berdasarkan KNN:
Laptop A - Rating: 4,8
Laptop A - Rating: 3,9
Raket Pro - Rating: 3,4
Topi C - Rating: 3,9
Keyboard Pro - Rating: 5
